# Lineborn Sales LoRA v2
Fresh fail-closed trainer for Qwen3-4B-Instruct-2507. This notebook will not produce or download an archive unless real SFT and DPO adapter weights exist. Use a GPU runtime.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(*args):
    print('>', ' '.join(map(str, args)), flush=True)
    return subprocess.run(list(map(str, args)), check=True)

run('nvidia-smi')
root = Path('/content/lineborn-runtime')
if root.exists():
    shutil.rmtree(root)
run('git', 'clone', '--depth', '1', '--branch', 'lineborn-sales-lora', 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git', str(root))
os.chdir(root)
run('git', 'rev-parse', 'HEAD')
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/requirements-colab.txt')
print('Environment ready:', root)

In [ ]:
run(sys.executable, 'training/build_sales_corpus.py')
run(sys.executable, 'training/validate_sales_corpus.py')

In [ ]:
run(sys.executable, 'training/train_sales_lora.py', '--max-length', '1536', '--epochs', '2', '--grad-accum', '16')
sft_model = Path('training/output/lineborn-sales-sft/adapter/adapter_model.safetensors')
assert sft_model.is_file(), 'SFT adapter_model.safetensors missing'
assert sft_model.stat().st_size > 1024 * 1024, f'SFT adapter suspiciously small: {sft_model.stat().st_size} bytes'
print(f'SFT adapter ready: {sft_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
run(sys.executable, 'training/train_sales_dpo.py', '--max-length', '1536', '--epochs', '1', '--grad-accum', '16')
dpo_model = Path('training/output/lineborn-sales-dpo/adapter/adapter_model.safetensors')
assert dpo_model.is_file(), 'DPO adapter_model.safetensors missing'
assert dpo_model.stat().st_size > 1024 * 1024, f'DPO adapter suspiciously small: {dpo_model.stat().st_size} bytes'
print(f'DPO adapter ready: {dpo_model.stat().st_size / 1024 / 1024:.1f} MiB')

In [ ]:
run(sys.executable, 'training/package_sales_adapters.py', '--require-dpo', '--archive', '/content/lineborn-sales-adapters-v2.zip')
archive = Path('/content/lineborn-sales-adapters-v2.zip')
assert archive.is_file(), 'candidate archive missing'
assert archive.stat().st_size > 2 * 1024 * 1024, f'candidate archive suspiciously small: {archive.stat().st_size} bytes'
print(f'Candidate archive ready: {archive.stat().st_size / 1024 / 1024:.1f} MiB')
from google.colab import files
files.download(str(archive))